# Trader AI Dashboard v1.5 | Decision Dashboard

本版聚焦每天真正有用的三个问题：当前风险环境是什么、由什么驱动、下一步重点观察什么。

输出包括：风险分层、相对强弱排名、一年百分位、综合仪表盘图与中文盘前摘要。

In [ ]:
# 安装行情获取和绘图依赖。首次运行约需几十秒。
%pip -q install yfinance pandas numpy matplotlib

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/cottonlyz-coder/trader-ai-dashboard.git'
COLAB_ROOT = Path('/content/trader-ai-dashboard')
candidates = [Path.cwd(), Path.cwd().parent, COLAB_ROOT]
PROJECT_ROOT = next((path for path in candidates if (path / 'src').exists()), None)

# 从 GitHub 直接打开 Colab 时，只会打开 Notebook 文件，因此自动下载完整项目。
if PROJECT_ROOT is None and Path('/content').exists():
    print('正在下载 Trader AI Dashboard 项目代码...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(COLAB_ROOT)], check=True)
    PROJECT_ROOT = COLAB_ROOT
elif PROJECT_ROOT == COLAB_ROOT and (COLAB_ROOT / '.git').exists():
    print('正在同步最新项目代码...')
    subprocess.run(['git', '-C', str(COLAB_ROOT), 'pull', '--ff-only'], check=True)

if PROJECT_ROOT is None or not (PROJECT_ROOT / 'src').exists():
    raise FileNotFoundError('未找到项目代码，请刷新运行环境后重试。')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'代码已就绪：{PROJECT_ROOT}')

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from src.dashboard import plot_decision_dashboard
from src.data_loader import load_market_data
from src.indicators import build_momentum_ranking, calculate_market_metrics
from src.regime import classify_market_regime
from src.report import build_dimension_table, generate_daily_brief_markdown

# 两年数据用于计算相对于过去一年的当前百分位位置。
prices, symbol_info, messages = load_market_data(period='2y')
metrics = calculate_market_metrics(prices)
regime = classify_market_regime(metrics)

display(Markdown(generate_daily_brief_markdown(metrics, regime)))

In [ ]:
display(Markdown('## 风险分层'))
display(build_dimension_table(regime).style.background_gradient(subset=['风险贡献分'], cmap='RdYlGn', vmin=-3, vmax=3))

display(Markdown('## 资产强弱排名 | 近20日'))
ranking = build_momentum_ranking(metrics)
display(ranking.style.format({'20D Change (%)': '{:+.2f}%', '50D Trend (%)': '{:+.2f}%', '1Y Level Percentile': '{:.0f}%'}).background_gradient(subset=['20D Change (%)'], cmap='RdYlGn'))

plot_decision_dashboard(prices, metrics, regime, lookback=120)
plt.show()

In [ ]:
display(Markdown('## 完整指标与数据质量记录'))
display(metrics.style.format({'Latest Price': '{:,.2f}', '1D Change (%)': '{:+.2f}%', '5D Change (%)': '{:+.2f}%', '20D Change (%)': '{:+.2f}%', '50D Trend (%)': '{:+.2f}%', '20D Volatility (%)': '{:.2f}%', '1Y Level Percentile': '{:.0f}%', '1Y Volatility Percentile': '{:.0f}%'}))
display(Markdown('### 实际使用的行情序列'))
display(symbol_info)
if messages:
    display(Markdown('### 数据提示'))
    for message in messages:
        print('-', message)